In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("model_data.csv")
df.head()

,Unnamed: 0,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY,NO2,Ozone,WS,WD,SR
0,0,2025-07-01 14:00:00,16.00,19.50,25.80,80.75,13.93,23.85,1.11,175.39,66.42
1,1,2025-07-01 15:00:00,16.01,19.55,25.82,80.83,15.38,21.56,0.80,176.21,40.23
2,2,2025-07-01 16:00:00,16.02,19.60,25.84,80.90,12.55,16.89,0.91,162.93,49.10
3,3,2025-07-02 08:00:00,16.17,20.38,26.15,82.14,7.32,16.68,0.67,154.23,9.45
4,4,2025-07-02 09:00:00,16.18,20.43,26.17,82.21,8.34,16.46,0.83,148.50,26.13


In [3]:
df = df.drop('Unnamed: 0', axis=1)

In [4]:
df.head()

,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY,NO2,Ozone,WS,WD,SR
0,2025-07-01 14:00:00,16.00,19.50,25.80,80.75,13.93,23.85,1.11,175.39,66.42
1,2025-07-01 15:00:00,16.01,19.55,25.82,80.83,15.38,21.56,0.80,176.21,40.23
2,2025-07-01 16:00:00,16.02,19.60,25.84,80.90,12.55,16.89,0.91,162.93,49.10
3,2025-07-02 08:00:00,16.17,20.38,26.15,82.14,7.32,16.68,0.67,154.23,9.45
4,2025-07-02 09:00:00,16.18,20.43,26.17,82.21,8.34,16.46,0.83,148.50,26.13


In [5]:
# AQI calculation
def calculate_aqi(row):
    return max(
        row['PM2.5'],
        row['PM10'] * 0.5,
        row['NO2'] * 0.8,
        row['Ozone'] * 0.6
    )

df['AQI'] = df.apply(calculate_aqi, axis=1)

In [7]:
print(df.dtypes)

TIMESTAMP       object
PM2.5          float64
PM10           float64
TEMPERATURE    float64
HUMIDITY       float64
NO2            float64
Ozone          float64
WS             float64
WD             float64
SR             float64
AQI            float64
dtype: object


In [9]:
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])

In [10]:
df['hour'] = df['TIMESTAMP'].dt.hour
df['day'] = df['TIMESTAMP'].dt.day
df['month'] = df['TIMESTAMP'].dt.month
df['day_of_week'] = df['TIMESTAMP'].dt.dayofweek

In [11]:
df = df.drop(columns=['TIMESTAMP'])

In [12]:
# Normalize data
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

scaled_df = pd.DataFrame(scaled_data, columns=df.columns)

STEP 2: Create LSTM Sequences (7 Days Input → 7 Days Output)

In [13]:
def create_sequences(data, n_past=7, n_future=7):
    X, y = [], []
    
    for i in range(len(data) - n_past - n_future):
        X.append(data[i:i+n_past])
        y.append(data[i+n_past:i+n_past+n_future])
    
    return np.array(X), np.array(y)

n_past = 7
n_future = 7

X, y = create_sequences(scaled_df.values, n_past, n_future)

print(X.shape)  # (samples, 7, features)
print(y.shape)  # (samples, 7, features)

(668, 7, 14)
(668, 7, 14)


STEP 3: Train-Test Split

In [14]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

STEP 4: Build LSTM Model

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(n_past, X.shape[2])),
    Dropout(0.2),
    
    LSTM(32, activation='relu'),
    Dropout(0.2),
    
    Dense(n_future * X.shape[2])  # flatten output
])

model.compile(optimizer='adam', loss='mse')

model.summary()

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 7, 64)               │          20,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 7, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 98)                  │           3,234 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 35,874 (140.13 KB)

 Trainable params: 35,874 (140.13 KB)

 Non-trainable params: 0 (0.00 B)

STEP 5: Train Model

In [17]:
history = model.fit(
    X_train, 
    y_train.reshape(y_train.shape[0], -1),
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test.reshape(y_test.shape[0], -1))
)

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.1589 - val_loss: 0.0758
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0787 - val_loss: 0.0567
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0583 - val_loss: 0.0473
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0475 - val_loss: 0.0411
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0432 - val_loss: 0.0396
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0403 - val_loss: 0.0374
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0372 - val_loss: 0.0363
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0355 - val_loss: 0.0359
Epoch 9/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0338 - val_loss: 0.0342
Epoch 10/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0320 - val_loss: 0.0321
Epoch 11/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0303 - val_loss: 0.0300
Epoch 12/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0294 -

STEP 6: Predict Next 7 Days

In [18]:
last_sequence = scaled_df.values[-n_past:]

last_sequence = last_sequence.reshape(1, n_past, X.shape[2])

pred = model.predict(last_sequence)

pred = pred.reshape(n_future, X.shape[2])

# Inverse transform
pred_actual = scaler.inverse_transform(pred)

pred_df = pd.DataFrame(pred_actual, columns=df.columns)

print(pred_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
       PM2.5       PM10  TEMPERATURE   HUMIDITY        NO2      Ozone  \
0  30.408709  45.474709    29.396889  83.113937  11.373266  19.307611   
1  31.326654  47.605011    28.787439  86.507393  11.317249  19.095369   
2  32.594830  47.515743    28.212332  87.295494  11.917374  19.041231   
3  31.881063  46.421677    28.032606  88.965996  11.660095  15.730483   
4  32.905403  47.777905    27.452091  90.930229  10.836571  13.548194   
5  32.944592  49.034431    27.578754  92.630890  11.808788  12.113538   
6  32.581600  48.955830    27.217215  90.926117  11.947194  10.905006   

         WS         WD          SR        AQI       hour        day  \
0  0.212492  44.384678  100.109756  33.153389  14.600928  12.084687   
1  0.243043  44.255516   70.612526  33.113087  14.010041  12.246900   
2  0.084249  45.176098   43.359493  33.069332  14.648687  11.711521   
3  0.302960  45.317173   27.908590  31.986320  12.894258  13.164618   
4  0.270031  46.286373  

Explainability Engine (WHY values changed)

In [19]:
def explain_changes(prev, curr):
    reasons = []

    if curr['WS'] < prev['WS']:
        reasons.append("Low wind speed → pollution accumulation")

    if curr['HUMIDITY'] > 70:
        reasons.append("High humidity → particles stay suspended")

    if curr['NO2'] > prev['NO2']:
        reasons.append("Increase in NO2 → combustion pollution")

    if curr['TEMPERATURE'] < prev['TEMPERATURE']:
        reasons.append("Lower temperature → poor dispersion")

    return reasons

Apply Explanation

In [20]:
explanations = []

for i in range(1, len(pred_df)):
    reasons = explain_changes(pred_df.iloc[i-1], pred_df.iloc[i])
    explanations.append(reasons)

In [27]:
pred_df.to_csv("pred_df.csv", index=False)

In [24]:
# Save full model
model.save("lstm_model4.h5")

# Save scaler (VERY IMPORTANT)
import joblib
joblib.dump(scaler, "scaler4.pkl")

['scaler4.pkl']

In [25]:
n_past = 7
n_future = 7

def predict_future(input_df):
    scaled = scaler.transform(input_df)
    
    last_seq = scaled[-n_past:]
    last_seq = last_seq.reshape(1, n_past, scaled.shape[1])
    
    pred = model.predict(last_seq)
    pred = pred.reshape(n_future, scaled.shape[1])
    
    pred_actual = scaler.inverse_transform(pred)
    
    pred_df = pd.DataFrame(pred_actual, columns=input_df.columns)
    
    return pred_df

In [28]:
feature_columns = df.columns.tolist()

import joblib
joblib.dump(feature_columns, "features.pkl")

['features.pkl']

In [29]:
def preprocess_input(df):
    # ❌ Drop unwanted columns
    df = df.drop(columns=['TIMESTAMP', 'Unnamed: 0'], errors='ignore')
    
    # ❗ If RF missing → add it
    if 'RF' not in df.columns:
        df['RF'] = 0   # or better: df['RF'].mean()
    
    # ❗ Remove extra column SR if not used in training
    if 'SR' not in feature_columns:
        df = df.drop(columns=['SR'], errors='ignore')
    
    # ✅ Ensure same order
    df = df[feature_columns]
    
    return df

In [30]:
class Preprocessor:
    def __init__(self, feature_columns):
        self.feature_columns = feature_columns
    
    def transform(self, df):
        df = df.drop(columns=['TIMESTAMP', 'Unnamed: 0'], errors='ignore')
        
        for col in self.feature_columns:
            if col not in df.columns:
                df[col] = 0
        
        df = df[self.feature_columns]
        return df